# 1. Dataset multiedición y validación

**Proyecto 2 MCC225** · Evaluación de CLIP-B, CLIP-L y LongCLIP sobre gráficos del IESF

Este notebook construye el manifiesto combinado de las tres ediciones del Informe
de Estabilidad del Sistema Financiero de la SBS y verifica su integridad antes de
cualquier evaluación.

**Por qué importa validar primero.** Un `image_id` duplicado o un caption repetido
dentro del mismo pool no produce un error visible: produce un resultado *plausible
pero inválido*. Es el tipo de fallo que aparece recién cuando alguien pregunta por
él en la defensa. Las pruebas de la sección 4 lo detienen antes.

## 1.1 Configuración

Se trabaja con rutas relativas al notebook para que el repositorio sea portable:
cualquiera que lo clone puede ejecutarlo sin editar rutas absolutas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# el notebook vive en notebooks/, el proyecto está un nivel arriba
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MULTI = BASE / "multiedicion"
RESULTS = BASE / "results"

MODELOS = ["CLIP", "CLIP-L", "LongCLIP"]
CAPTIONS = ["caption_2", "caption_3", "caption_4"]

import sys
sys.path.insert(0, str(BASE))
from config_variante import VARIANTE, carpeta_imagenes, ruta_imagen, resumen

print("proyecto:", BASE)
print(resumen(MULTI))
print("existe multiedicion/:", MULTI.exists())
print("existe results/:", RESULTS.exists())

## 1.2 Unir las tres ediciones

El manifiesto original (`manifest_local2.csv`, edición 2026-1) se combina con los
dos derivados de las ediciones 2024-2 y 2021-1.

Se agrega `chart_uid = edición|chart_id` porque **los identificadores de gráfico se
repiten entre ediciones**: `I.A.1` existe tanto en 2021 como en 2024. Sin ese
identificador compuesto, cualquier agrupación posterior mezclaría gráficos
distintos.

In [ ]:
COMBINADO = MULTI / "manifest_multiedicion.csv"

if COMBINADO.exists():
    # fuente única: ya contiene las tres ediciones con todas las columnas
    man = pd.read_csv(COMBINADO)
    print(f"manifiesto combinado cargado: {len(man)} pares")
else:
    # reconstrucción desde los manifiestos por edición.
    # El patrón manifest_20*.csv es deliberado: toma manifest_2021-1,
    # manifest_2024-2 y manifest_2026-1, y deja fuera cualquier copia con otro
    # nombre que duplicaría filas de una misma edición.
    partes = [pd.read_csv(f) for f in sorted(MULTI.glob("manifest_20*.csv"))]
    if not partes:
        raise FileNotFoundError(f"no hay manifiestos por edición en {MULTI}")
    man = pd.concat(partes, ignore_index=True)
    man = man.drop_duplicates(subset="image_id", keep="first")
    man.to_csv(COMBINADO, index=False, encoding="utf-8")
    print(f"manifiesto combinado construido: {len(man)} pares")

print(man.groupby("edicion").size().to_string())

# aviso si hay copias sueltas que podrían confundir
sueltos = [f.name for f in MULTI.glob("manifest_*.csv")
           if f.name not in {"manifest_multiedicion.csv"}
           and not f.name.startswith(("manifest_2021", "manifest_2024", "manifest_2026"))]
if sueltos:
    print(f"\naviso: manifiestos con nombre no estándar (no se usan): {sueltos}")

## 1.3 Colisiones de títulos entre ediciones

Aquí aparece la razón de la decisión de diseño más importante del experimento.

Algunos gráficos conservan el mismo título entre ediciones. Si las 95 imágenes
compitieran en un solo pool de recuperación, esos pares **no tendrían respuesta
correcta única** y el modelo sería penalizado por un acierto legítimo.

Por eso **cada edición se evalúa como un pool independiente**. Hay una segunda
razón, igual de importante: el R@1 depende del número de distractores, así que un
pool de 95 no sería comparable con el de 40 ya reportado — una caída no
significaría peor modelo, sino tarea más difícil.

In [ ]:
tit = man["caption_2"].str.lower().str.strip()
colisiones = man.groupby(tit)["edicion"].nunique()
colisiones = colisiones[colisiones > 1]

print(f"títulos repetidos entre ediciones: {len(colisiones)}\n")
for t in colisiones.index:
    sub = man.loc[tit == t, ["edicion", "chart_id"]]
    print(f"  {t[:60]}")
    for _, r in sub.iterrows():
        print(f"      {r.edicion}  {r.chart_id}")

## 1.4 Confound declarado: cómo cita cada edición sus gráficos

Los `caption_3` y `caption_4` se construyen con la oración del informe que comenta
el gráfico. **La forma de citar cambió con el tiempo**: desde 2026 la SBS
referencia los gráficos explícitamente en el cuerpo ("(Gráfico I.1)"), mientras
que en 2021 y 2024 los inserta después del párrafo sin citarlos.

Esto es una variable de confusión *entre* ediciones: si 2026 rinde mejor, podría
deberse a que sus captions largos están mejor alineados con el gráfico y no a
diferencias del modelo.

Se controla de dos maneras: comparando modelos **dentro** de cada edición, donde
la fuente es constante para los tres, y reportando el R@1 desagregado por
`fuente_oracion`.

In [ ]:
print(pd.crosstab(man["edicion"], man["fuente_oracion"]).to_string())

## 1.5 Pruebas de integridad

Nueve verificaciones. La más sutil es la de duplicados: se evalúa **dentro de cada
edición**, no globalmente, porque entre pools separados un título repetido es
inofensivo — los gráficos nunca compiten entre sí.

In [ ]:
def test_numero_de_pares():
    assert len(man) >= 90, f"se esperaban ~95 pares, hay {len(man)}"

def test_image_id_unicos():
    dup = man["image_id"][man["image_id"].duplicated()].tolist()
    assert not dup, f"image_id duplicados: {dup}"

def test_chart_uid_unicos():
    dup = man["chart_uid"][man["chart_uid"].duplicated()].tolist()
    assert not dup, f"chart_uid duplicados: {dup}"

def test_imagenes_existen():
    faltan = [p for p in man["image_path"]
              if not Path(ruta_imagen(MULTI, p)).exists()]
    assert not faltan, f"no existen {len(faltan)} imágenes: {faltan[:5]}"

def test_imagenes_no_vacias():
    vac = [p for p in man["image_path"]
           if Path(ruta_imagen(MULTI, p)).exists()
           and Path(ruta_imagen(MULTI, p)).stat().st_size == 0]
    assert not vac, f"imágenes de 0 bytes: {vac}"

def test_captions_no_vacios():
    prob = []
    for cap in CAPTIONS:
        v = man[man[cap].isna() | (man[cap].astype(str).str.strip() == "")]
        if len(v):
            prob.append(f"{cap}: {len(v)} vacíos")
    assert not prob, prob

def test_captions_sin_duplicados_dentro_de_edicion():
    prob = []
    for ed, g in man.groupby("edicion"):
        for cap in CAPTIONS:
            vc = g[cap].astype(str).str.strip().value_counts()
            if (vc > 1).any():
                prob.append(f"{ed}/{cap}: {int((vc > 1).sum())} repetidos")
    assert not prob, prob

def test_pools_suficientes():
    chicos = {ed: len(g) for ed, g in man.groupby("edicion") if len(g) < 20}
    assert not chicos, f"pools demasiado pequeños: {chicos}"

def test_rutas_relativas():
    abs_ = [p for p in man["image_path"] if str(p).startswith("/") or ":" in str(p)[:3]]
    assert not abs_, f"rutas absolutas: {abs_[:3]}"


pruebas = [v for k, v in sorted(globals().items())
           if k.startswith("test_") and callable(v)]
fallos = 0
for t in pruebas:
    try:
        t()
        print(f"  OK    {t.__name__}")
    except AssertionError as e:
        fallos += 1
        print(f"  FALLA {t.__name__}: {e}")

print(f"\n{len(pruebas) - fallos}/{len(pruebas)} pruebas pasaron")

## 1.6 Inspección visual

Las métricas no detectan un recorte mal hecho. Conviene mirar una muestra de cada
edición antes de la corrida definitiva: si el párrafo del informe quedara visible
dentro de la imagen, el modelo podría emparejarla con su caption **leyendo ese
texto** en vez de interpretando el gráfico, lo que inflaría el R@1 por una vía
distinta de la que se quiere medir.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

muestra = man.groupby("edicion").head(1)
fig, axes = plt.subplots(len(muestra), 1, figsize=(11, 5 * len(muestra)))
axes = np.atleast_1d(axes)

for ax, (_, r) in zip(axes, muestra.iterrows()):
    ax.imshow(Image.open(ruta_imagen(MULTI, r.image_path)))
    ax.set_title(f"{r.edicion} — {r.chart_id}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 1.7 Resumen

| | |
|---|---|
| Pares totales | ver salida de 1.2 |
| Pools | uno por edición, independientes |
| Identificador único | `chart_uid` = edición\|chart_id |
| Confound declarado | `fuente_oracion` difiere entre ediciones |

El manifiesto combinado queda en `multiedicion/manifest_multiedicion.csv` y es la
entrada del notebook 3.